# Evaluating LM Outputs using Rubric + LM Judges

Instead of using BLADE's `EntireAnalysisProcessed` code, we will try an evaluation implementation that reads straight from `multirun_analyses.json` and gives the results to an LLM judge.

## Setup

In [1]:
# imports
import json
import pandas as pd
import importlib.util
import sys
from os.path import join
from stat_genie.blade_pipeline.llms.config import llm
from stat_genie.blade_pipeline.additions.eval.extraction import \
    format_features, format_model_info
from stat_genie.blade_pipeline.additions.analysis.conclusion import \
    write_final_answer_code, make_conclusion

In [2]:
# define file paths
analysis_subdir_path = "analysis_output"
multirun_filename = "multirun_analyses.json"
# use multirun analyses file to get analysis code paths
multirun_analyses_path = join(analysis_subdir_path, multirun_filename)
with open(multirun_analyses_path, "r") as file:
    multirun_analyses = json.load(file)
num_analyses = multirun_analyses['n']
analysis_code_filenames = [f"llm_analysis_{i}.py" for i in range(num_analyses)]
analysis_code_paths = [join(analysis_subdir_path, filename) for filename in \
    analysis_code_filenames]
# get config details
llm_provider = "openai"
llm_model = "gpt-5-mini"
# create llm assistant
llm_assistant = llm(provider=llm_provider, model=llm_model)

[2025-11-21 03:40:34.50][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/projects/binyu/hao_huang/stat-genie/config/llm_eval_config.yml'.


## Extract Features **X** Used in Model

In [3]:
# create dict to store features
# features = {}

In [4]:
# # loop through analyses
# for i, analysis_code_path in enumerate(analysis_code_paths):
    
#     # create internal dict for analysis features
#     features[i] = {}
    
#     # get the features from each analysis
#     # this should include the independent and control variables
#     ind_vars = multirun_analyses['analyses'][str(i)]['cvars']['ivs']
#     control_vars = multirun_analyses['analyses'][str(i)]['cvars']['controls']

#     # get any lines from the transform code that represent transformations
#     # of the independent or control variables
#     transform_code = multirun_analyses['analyses'][str(i)]['transform_code']
#     # for each variable in ind_vars, check if it is transformed
#     # in the transform_code by using an LLM assistant
#     llm_assistant = llm(provider=llm_provider, model=llm_model)
#     for dict_idx, var in enumerate(ind_vars):
#         transform_responses = get_feature_transforms(llm_assistant,
#                                                      transform_code,
#                                                      var['columns'],
#                                                      var['description'])
#         ind_vars[dict_idx]['transform_code'] = [response.text[0].content for \
#             response in transform_responses]
        
#     # save updated independent variables in features dict
#     features[i]['independent_variables'] = ind_vars
    
#     # tkae same approach for control variables
#     for dict_idx, var in enumerate(control_vars):
#         transform_responses = get_feature_transforms(llm_assistant,
#                                                      transform_code,
#                                                      var['columns'],
#                                                      var['description'])
#         control_vars[dict_idx]['transform_code'] = [response.text[0].content for \
#             response in transform_responses]
    
#     # save updated control variables in features dict
#     features[i]['control_variables'] = control_vars

In [5]:
# view feature dictionary to ensure correctness
# features

## Extract Response *y* used in Model

In [6]:
# # loop through analyses
# for i, analysis_code_path in enumerate(analysis_code_paths):
    
#     # get the features from each analysis
#     # this should include the independent and control variables
#     response_vars = multirun_analyses['analyses'][str(i)]['cvars']['dv']

#     # get any lines from the transform code that represent transformations
#     # of the independent or control variables
#     transform_code = multirun_analyses['analyses'][str(i)]['transform_code']
#     # for each variable in response_vars, check if it is transformed
#     # in the transform_code by using an LLM assistant
#     llm_assistant = llm(provider=llm_provider, model=llm_model)
#     # for dict_idx, var in enumerate(response_vars):
#     transform_responses = get_feature_transforms(llm_assistant,
#                                                  transform_code,
#                                                  response_vars['columns'],
#                                                  response_vars['description'])
#     response_vars['transform_code'] = [response.text[0].content \
#         for response in transform_responses]

#     # save updated response variables in features dict
#     features[i]['response_variables'] = response_vars

In [7]:
# view feature dictionary to ensure correctness
# features

## Extract Features **X** and *y* Used in Model

In [8]:
features = format_features(multirun_analyses, num_analyses, llm_assistant)

[2025-11-21 03:40:35.43][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-21 03:40:44.16][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  8.72 seconds
[2025-11-21 03:40:44.16][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-21 03:40:44.19][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-21 03:40:53.30][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  9.11 seconds
[2025-11-21 03:40:53.31][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-21 03:40:53.33][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-21 03:40:58.31][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  4.98 

In [9]:
features

{0: {'independent_variables': [{'description': 'Continuous masculinity-femininity name score (higher = more feminine). Primary independent variable: hypothesis predicts higher femininity (higher masfem) -> fewer precautions -> greater human cost (deaths). This is standardized (z-scored) in the transformed dataframe.',
    'columns': ['masfem_z'],
    'transform_code': ["if 'masfem' in df.columns:\n    df['masfem_z'] = (df['masfem'] - df['masfem'].mean()) / (df['masfem'].std(ddof=0) if df['masfem'].std(ddof=0) != 0 else 1)\nelse:\n    df['masfem_z'] = np.nan"]},
   {'description': 'Alternative continuous masculinity-femininity score collected on MTurk (independent replication of name femininity). Standardized in the transformed dataframe.',
    'columns': ['masfem_mturk_z'],
    'transform_code': ["if 'masfem_mturk' in df.columns:\n    df['masfem_mturk_z'] = (df['masfem_mturk'] - df['masfem_mturk'].mean()) / (df['masfem_mturk'].std(ddof=0) if df['masfem_mturk'].std(ddof=0) != 0 else 1)\

## Extract Model Class Used

In [10]:
model_info = format_model_info(multirun_analyses, num_analyses, llm_assistant)
model_info

[2025-11-21 03:43:21.28][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)


[2025-11-21 03:43:37.50][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  16.22 seconds
[2025-11-21 03:43:37.50][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-21 03:43:37.52][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-21 03:43:49.08][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  11.57 seconds
[2025-11-21 03:43:49.09][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)


{0: '{\n  "model_library": "statsmodels (imported as statsmodels.api, alias sm)",\n  "model_class": "sm.OLS (Ordinary Least Squares linear regression)",\n  "model_parameters": "missing=\'drop\' passed to sm.OLS; intercept added via sm.add_constant(X); X and y cast to float via .astype(float); .fit() called with default OLS settings (no robust covariance specified)",\n  "model_formula_fitting_code": "# Main model: LogDeaths ~ masfem_z + controls\\nX_main_cols = [\'masfem_z\'] + control_cols\\nX_main = df[X_main_cols].astype(float)\\nX_main = sm.add_constant(X_main)\\ny_main = df[\'LogDeaths\'].astype(float)\\nmodel_main = sm.OLS(y_main, X_main, missing=\'drop\').fit()\\nresults[\'main_model\'] = model_main\\n\\n# Robustness 1: LogDeaths ~ gender_female + controls\\nX_g_cols = [\'gender_female\'] + control_cols\\nX_g = df[X_g_cols].astype(float)\\nX_g = sm.add_constant(X_g)\\ny_g = df[\'LogDeaths\'].astype(float)\\nmodel_gender = sm.OLS(y_g, X_g, missing=\'drop\').fit()\\nresults[\'gende

## Extract Final Answer/Conclusion

Each of the BLADE tasks revolves around a question with the following format:

*What is the effect of [something] on [potential response]?*

It seems that often times the feature to use for the response is not deterministic; the model will have to use some sort of proxy to estimate it. The explanatory features are typically a little bit more clear, but still often require transformations and judgment calls on interpretation and use.

Importantly, this type of question ensures there is a binary answer. While the LLM data scientist does not explicitly spit out a yes/no value, it does write two functions: one which preprocesses the data and another that performs some sort of analysis. Theoretically, we could take the output of the analysis and inspect it to determine whether or not the feature of interest had an effect on the response.

In [11]:
# get path to the dataset
dataset_name = multirun_analyses['dataset_name']
dataset_path = join("..", "..", "src", "stat_genie", "blade_pipeline",
                    "datasets", dataset_name, "data.csv")

# load the dataset
data = pd.read_csv(dataset_path)

# create dictionaries to store the imported functions
transform_functions = {}
model_functions = {}

# loop through analyses
for i, analysis_code_path in enumerate(analysis_code_paths):
    # dynamically import the module
    spec = importlib.util.spec_from_file_location(f"llm_analysis_{i}",
                                                  analysis_code_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f"llm_analysis_{i}"] = module
    spec.loader.exec_module(module)
    
    # extract transform and model functions
    transform_functions[i] = module.transform
    model_functions[i] = module.model

In [12]:
# Run transform functions on the dataset
transformed_datasets = {}
for i, transform_func in transform_functions.items():
    try:
        transformed_datasets[i] = transform_func(data.copy())  # use copy of dataset
        print(f"[Transform {i}] ✅ Completed successfully.")
    except Exception as e:
        # print(f"[Transform {i}] ❌ Failed with error: {e}")
        print(f"[Transform {i}] ❌")
        transformed_datasets[i] = None

# Run model functions on the transformed datasets
model_results = {}
for i, model_func in model_functions.items():
    try:
        if transformed_datasets[i] is None:
            print(f"[Model {i}] ⚠️ Skipping — transform step failed.")
            continue

        model_results[i] = model_func(transformed_datasets[i].copy())  # use copy
        print(f"[Model {i}] ✅ Completed successfully.")
    except Exception as e:
        print(f"[Model {i}] ❌ Failed with error: {e}")
        model_results[i] = None


[Transform 0] ✅ Completed successfully.
[Transform 1] ✅ Completed successfully.
[Model 0] ✅ Completed successfully.
[Model 1] ✅ Completed successfully.


/accounts/projects/binyu/hao_huang/stat-genie/.venv/lib/python3.11/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


In [13]:
# view the first model result as a sanity check
model_results[0]

{'main_model': <statsmodels.regression.linear_model.RegressionResultsWrapper at 0x72e0914a0a50>,
 'gender_model': <statsmodels.regression.linear_model.RegressionResultsWrapper at 0x72e09076f4d0>,
 'damage_model': <statsmodels.regression.linear_model.RegressionResultsWrapper at 0x72e09077d790>}

In [14]:
# create storage object for final answers
final_answer_code = {}

# read task from info.json in the dataset directory
info_json_path = join("..", "..", "src", "stat_genie", "blade_pipeline",
                      "datasets", dataset_name, "info.json")
with open(info_json_path, "r") as file:
    info_json = json.load(file)
task = info_json['research_questions']

for i in range(num_analyses):

    # get the independent and dependent variables from the dictionary made in
    # previous cells
    independent_variable = features[i]['independent_variables']
    dependent_variable = features[i]['response_variables']

    # get the model code
    model_code = multirun_analyses['analyses'][str(i)]['m_code']
    
    # get the model output from object made in previous cell
    model_output = model_results[i]
    
    # call the helper function
    final_answer_code[i] = write_final_answer_code(llm_assistant, task,
                                                   independent_variable,
                                                   dependent_variable,
                                                   model_code, model_output)

[2025-11-21 03:43:58.26][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)


[2025-11-21 03:44:25.70][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  27.44 seconds
[2025-11-21 03:44:25.71][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-21 03:44:25.73][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-21 03:44:56.74][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  31.00 seconds
[2025-11-21 03:44:56.74][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)


In [15]:
# run the final answer code
final_answer_code

{0: 'def extract_final_answer(model_output):\n    """\n    Extract key statistics from the supplied statsmodels fitted model objects and\n    produce a concise conclusion about the hypothesis:\n      "More feminine hurricane names -> fewer precautions -> greater human cost (higher LogDeaths)."\n\n    Parameters\n    ----------\n    model_output : dict\n        Expected keys: \'main_model\', \'gender_model\', \'damage_model\'\n        Values are statsmodels RegressionResultsWrapper objects (or None).\n\n    Returns\n    -------\n    dict with keys:\n      - "object": dict summarizing coefficients, SE, t, p, 95% CI, n for each model and a boolean\n                  \'supports_hypothesis\' based on the main_model (coef>0 and p<0.05).\n      - "description": short explanation of what the extracted numbers mean in context.\n    """\n    def _summarize_model(model, varname):\n        if model is None:\n            return None\n        try:\n            params = model.params\n        except E

In [16]:
# loop through final answer code and dynamically execute the functions
final_answer_functions = {}
for i in range(num_analyses):
    # create a namespace dictionary to execute the code in
    namespace = {}
    
    # compile and execute the code
    compiled_code = compile(final_answer_code[i], f"<final_answer_code_{i}>", "exec")
    exec(compiled_code, namespace)
    
    # extract function from namespace
    final_answer_functions[i] = namespace['extract_final_answer']

# run the final answer functions on the model results
final_answers = [final_answer_functions[i](model_results[i]) for i in range(num_analyses)]

In [17]:
conclusions = {}
for i in range(num_analyses):

    # get the independent and dependent variables from the dictionary made in
    # previous cells
    independent_variable = features[i]['independent_variables']
    dependent_variable = features[i]['response_variables']

    # get the model code
    model_code = multirun_analyses['analyses'][str(i)]['m_code']
    
    # get the model interpretation code
    try:
        interpretation_code = final_answer_code[i]
    except (KeyError, IndexError, TypeError):
        interpretation_code = None
    
    try: 
        interpretation_output = final_answers[i]
    except (KeyError, IndexError, TypeError):
        interpretation_output = None
    
    # call the helper function
    conclusions[i] = make_conclusion(llm_assistant, task, independent_variable,
                                     dependent_variable, model_code,
                                     interpretation_code, interpretation_output)

[2025-11-21 03:44:57.04][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-21 03:45:01.72][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  4.68 seconds
[2025-11-21 03:45:01.73][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-21 03:45:01.75][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-21 03:45:06.85][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  5.09 seconds
[2025-11-21 03:45:06.85][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)


In [18]:
conclusions

{0: '{\n  "answer": "No",\n  "justification": "The main model estimates a small positive effect (coef = 0.0818) but it is not statistically significant (p = 0.5447; 95% CI = [-0.186, 0.349]). Robustness checks (binary gender and LogDamage) are also non‑significant, so the analysis does not support the hypothesis."\n}',
 1: '{\n  "answer": "No",\n  "justification": "The masfem_z coefficient is positive but not statistically significant (coef=0.1417, p=0.704; 95% CI includes 0) and the IRR CI includes 1. The binary gender_mf predictor is also non-significant. Thus the model provides no evidence that more-feminine names are associated with higher fatalities (fewer precautions)."\n}'}

In [19]:
llm_judge = llm(provider=llm_provider, model=llm_model)

[2025-11-21 03:45:06.99][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/projects/binyu/hao_huang/stat-genie/config/llm_eval_config.yml'.


In [20]:
data_head = data.head(10)

In [21]:
task

['Hurricanes with more feminine names are perceived as less threatening and hence lead to fewer precautionary measures by the general public.']

In [22]:
judge_system_prompt = (
    "You are a meticulous research design evaluator. "
    "Your role is to compare two experimental trials methodologically **and interpretively**.\n\n"
    "You will go through the following reasoning plan step-by-step (internally):\n"
    "1. Understand the research question and dataset context.\n"
    "2. Examine independent, control, and response variables for both trials.\n"
    "3. Analyze the model specifications for structural or methodological similarity.\n"
    "4. Focus more on the content, less on the format.\n"
    "5. Assess whether the trials' conclusions are logically consistent given their setups.\n"
    "6. Detect whether either input is None, invalid, erroneous, or incomplete.\n"
    "   - If **one trial** shows errors or missing components but the other is valid, "
    "     impose a **strong penalty** (reduce all category scores by at least 1 point, "
    "     and cap overall similarity at 2).\n"
    "7. Synthesize your evaluation across all components.\n"
    "8. Output a numerical rating for each category.\n\n"
    "DO NOT include your reasoning — only the final JSON object.\n\n"
    "Scoring scale:\n"
    "1 = completely different\n"
    "2 = somewhat different\n"
    "3 = moderately similar\n"
    "4 = very similar\n"
    "5 = almost identical\n\n"
    "Return output **strictly in JSON format**:\n"
    "{\n"
    "  \"independent_variables\": <number>,\n"
    "  \"control_variables\": <number>,\n"
    "  \"response_variables\": <number>,\n"
    "  \"model_specification\": <number>,\n"
    "  \"conclusions\": <number>,\n"
    "  \"overall_similarity\": <number>\n"
    "}"
)

In [23]:
judge_user_prompt = (
    f"Research Question / Context:\n{task}\n\n"
    "Here is a sample of the dataset to understand the structure and variables:\n"
    f"{data_head}\n\n"
    "Compare the two trials methodologically and interpretively based on the provided variables, model specifications, and conclusions.\n\n"
    "==================== TRIAL 0 ====================\n\n"
    "Independent Variables:\n"
    f"{features[0]['independent_variables']}\n\n"
    "Control Variables:\n"
    f"{features[0]['control_variables']}\n\n"
    "Response Variables:\n"
    f"{features[0]['response_variables']}\n\n"
    "Model Specification:\n"
    f"{model_info[0]}\n\n"
    "Conclusion:\n"
    f"{conclusions[0]}\n\n"
    "==================== TRIAL 1 ====================\n\n"
    "Independent Variables:\n"
    f"{features[1]['independent_variables']}\n\n"
    "Control Variables:\n"
    f"{features[1]['control_variables']}\n\n"
    "Response Variables:\n"
    f"{features[1]['response_variables']}\n\n"
    "Model Specification:\n"
    f"{model_info[1]}\n\n"
    "Conclusion:\n"
    f"{conclusions[1]}\n\n"
    "Now, following your reasoning plan, provide similarity ratings as JSON only."
)

In [24]:
final_scores = llm_judge.generate([{"role": "system",
                                        "content": judge_system_prompt},
                                       {"role": "user",
                                        "content": judge_user_prompt}])

[2025-11-21 03:45:07.84][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)


[2025-11-21 03:45:17.51][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  9.67 seconds
[2025-11-21 03:45:17.52][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)


In [25]:
final_scores.text[0].content

'{\n  "independent_variables": 4,\n  "control_variables": 4,\n  "response_variables": 3,\n  "model_specification": 2,\n  "conclusions": 5,\n  "overall_similarity": 3\n}'